# LAB I: Using MCP in LangChain

This notebook provides a starter setup for connecting LangChain to OpenAI using environment variables and MCP-style integration.

In [ ]:
#pip install langchain-mcp-adapters mcp

Note: you may need to restart the kernel to use updated packages.


In [42]:
from pathlib import Path
from dotenv import load_dotenv
import os
import mcp
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import BaseTool

load_dotenv(Path('.') / '.env')
openai_key = os.getenv('OPENAI_API_KEY')
print('OPENAI_API_KEY is set:', bool(openai_key))


OPENAI_API_KEY is set: True


In [43]:
!npx --version

11.12.1


In [44]:
# %%
client = MultiServerMCPClient({
    "my_server": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/private/tmp"],
        "transport": "stdio"
    }
})

tools = await client.get_tools()
print(f"{len(tools)} Tool(s) gefunden:")
for t in tools:
    print(f"  - {t.name}: {t.description}")


14 Tool(s) gefunden:
  - read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
  - read_text_file: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
  - read_media_file: Read an image or audio file. Returns the base64 encoded data and MIME type. Only works within allowed directories.
  - read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you need to analyze or compare multiple files. Each file's content is returned with its path as a reference. Failed reads for ind

In [45]:
# %%
print(f"Tools sind LangChain-kompatibel: {all(isinstance(t, BaseTool) for t in tools)}")


Tools sind LangChain-kompatibel: True


In [46]:

# %%
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)
agent = create_react_agent(llm, tools)

result = await agent.ainvoke({
    "messages": [{"role": "user", "content": "List the files in /private/tmp and tell me what's there."}]
})

print(result["messages"][-1].content)

/var/folders/2g/5q_lj5g51f19wyt7z8k7qxs00000gn/T/ipykernel_6781/1699540669.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In the directory `/private/tmp`, the following files and directories are present:

### Files
- **Visual Studio Code-afd45a32-d052-4a3c-bc1f-37c9dc13f7e9.sock**
- **zeb_def_ipc_1197**

### Directories
- **claude-501**
- **com.apple.launchd.AtbLPtkApO**
- **node-compile-cache**
- **powerlog** 

If you need more details about any specific item, feel free to ask!


In [48]:
# %% [markdown]
# ## Step 5: MCP Resources
# Der @modelcontextprotocol/server-filesystem unterstützt keine Resources (nur Tools).
# Resources sind ein optionales MCP-Feature — nicht alle Server implementieren es.
# Beispiele für Resource-fähige Server: Datenbank-Server, Dokumenten-Server.

# %%
try:
    resources = await client.get_resources()
    print(f"{len(resources)} Resource(s) gefunden:")
    for r in resources:
        print(f"  - {r.uri}: {r.name}")
except Exception as e:
    print(f"Dieser Server unterstützt keine Resources: {type(e).__name__}")
    print("Resources sind ein optionales MCP-Feature.")

Dieser Server unterstützt keine Resources: ExceptionGroup
Resources sind ein optionales MCP-Feature.


In [49]:
# %% [markdown]
# ## Step 6: Complete MCP-Enabled Agent

# %%
agent = create_react_agent(llm, tools)

test_queries = [
    "What files and directories exist in /private/tmp?",
    "Create a file called test_mcp.txt in /private/tmp with the content 'MCP works!'",
    "Read the file /private/tmp/test_mcp.txt and tell me its content.",
]

for query in test_queries:
    print(f"\n--- Query: {query} ---")
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": query}]
    })
    print(result["messages"][-1].content)
    print()

/var/folders/2g/5q_lj5g51f19wyt7z8k7qxs00000gn/T/ipykernel_6781/3265555394.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)



--- Query: What files and directories exist in /private/tmp? ---
In the `/private/tmp` directory, the following files and directories exist:

### Files:
- `Visual Studio Code-afd45a32-d052-4a3c-bc1f-37c9dc13f7e9.sock`
- `zeb_def_ipc_1197`

### Directories:
- `claude-501`
- `com.apple.launchd.AtbLPtkApO`
- `node-compile-cache`
- `powerlog`


--- Query: Create a file called test_mcp.txt in /private/tmp with the content 'MCP works!' ---
The file `test_mcp.txt` has been successfully created in the `/private/tmp` directory with the content "MCP works!".


--- Query: Read the file /private/tmp/test_mcp.txt and tell me its content. ---
The content of the file `/private/tmp/test_mcp.txt` is: **"MCP works!"**

